In [ ]:
import numpy as np
import cv2
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rawatjitesh/avengers-face-recognition")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'avengers-face-recognition' dataset.
Path to dataset files: /kaggle/input/avengers-face-recognition


In [ ]:
import os

print(os.listdir(path))

['cropped_images']


In [ ]:
data = []
labels = []

# Explicitly set path to the correct downloaded dataset location
path = '/root/.cache/kagglehub/datasets/rawatjitesh/avengers-face-recognition/versions/1'
dataset_path = os.path.join(path, 'cropped_images')
classes = os.listdir(dataset_path)

for label, folder in enumerate(classes):
    folder_path = os.path.join(dataset_path, folder)

    for img_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_name)

        img = cv2.imread(img_path)
        img = cv2.resize(img, (100,100))

        data.append(img)
        labels.append(label)

In [ ]:
# Convert to Array
X = np.array(data) / 255.0
y = np.array(labels)

In [ ]:
# One-Hot Encoding
y = np.array(labels)
y = to_categorical(y, len(classes))

In [ ]:
# Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# build CNN Model
model = Sequential()

model.add(Conv2D(32, (3,3), activation='relu', input_shape=(100,100,3)))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D(2,2))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dense(len(classes), activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# Compile Model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Train Model
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 567ms/step - accuracy: 0.1943 - loss: 2.9235 - val_accuracy: 0.1818 - val_loss: 1.6828
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.2400 - loss: 1.6132 - val_accuracy: 0.3864 - val_loss: 1.5813
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4000 - loss: 1.4949 - val_accuracy: 0.5000 - val_loss: 1.5264
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5429 - loss: 1.3580 - val_accuracy: 0.5682 - val_loss: 1.3066
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7429 - loss: 1.1341 - val_accuracy: 0.7045 - val_loss: 1.0883
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7486 - loss: 0.9086 - val_accuracy: 0.7955 - val_loss: 0.8465
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.8457 - loss: 0.6309 - val_accuracy: 0.7500 - val_loss: 0.7023
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9429 - loss: 0.3760 - val_accuracy: 0.7273 - val_loss: 0.6730

In [ ]:
# Evaluate
loss, acc = model.evaluate(X_test, y_test)
print("Accuracy:", acc)

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 590ms/step - accuracy: 0.6727 - loss: 0.8577
Accuracy: 0.6727272868156433


In [ ]:
# Prediction
img = cv2.imread("test.jpg")

if img is None:
    print("Image not found ❌")
else:
    img = cv2.resize(img, (100,100)) / 255.0
    img = np.reshape(img, (1,100,100,3))

Image not found ❌


In [ ]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)